# Teachable Machine — CNN

Entrene y pruebe modelos de clasificación de imágenes, todo interactivo.

| Paso | Descripción |
|------|-------------|
| 1. Datos | Cargar carpeta de imágenes O capturar con cámara |
| 2. Entrenar | Un clic — o cargar modelo .h5 existente |
| 3. Probar | Evaluar dataset, probar imágenes individuales |
| 4. Exportar | Guardar .h5 + config para cámara en vivo |

> Ejecute las celdas en orden. Los botones hacen el trabajo.

In [ ]:
import os, sys, glob, random, shutil, io, time, subprocess
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output

random.seed(42); np.random.seed(42); tf.random.set_seed(42)

_S = {
    'model': None, 'class_names': [], 'img_size': 128,
    'history': None, 'dataset_dir': '', 'split_dir': ''
}

print(f"TensorFlow {tf.__version__} — Listo")

---
## 1. Datos

Elija cómo obtener las imágenes para entrenar:

- **Carpeta**: Si ya tiene imágenes organizadas en subcarpetas por clase
- **Cámara**: Captura imágenes directamente desde la cámara (tiempo fijo por clase para mantener balance)

In [ ]:
# ========= TAB 1: CARPETA =========
w_data_path = widgets.Text(
    value='../data/tapitas', placeholder='Ruta a carpeta con subcarpetas por clase',
    description='Ruta:', layout=widgets.Layout(width='500px')
)
w_imgsize = widgets.Dropdown(
    options=[64, 128, 224], value=128, description='Resoluci\u00f3n:'
)
w_btn_folder = widgets.Button(
    description=' Cargar carpeta', button_style='primary', icon='folder-open'
)

# ========= TAB 2: C\u00c1MARA =========
w_cam_classes = widgets.Text(
    value='', placeholder='rojo, azul, verde',
    description='Clases:', layout=widgets.Layout(width='500px')
)
w_cam_time = widgets.IntSlider(
    value=10, min=5, max=30, step=5,
    description='Seg/clase:', layout=widgets.Layout(width='400px')
)
w_cam_output = widgets.Text(
    value='../data/mi_dataset', placeholder='Directorio de salida',
    description='Guardar en:', layout=widgets.Layout(width='500px')
)
w_cam_index = widgets.IntText(value=0, description='C\u00e1mara #:',
                               layout=widgets.Layout(width='200px'))
w_btn_camera = widgets.Button(
    description=' Abrir c\u00e1mara', button_style='warning', icon='video-camera'
)

w_imgsize2 = widgets.Dropdown(
    options=[64, 128, 224], value=128, description='Resoluci\u00f3n:'
)

w_out_data = widgets.Output()

def _show_dataset_info(ddir, img_size):
    """Muestra resumen del dataset."""
    clases = sorted([d for d in os.listdir(ddir)
                     if os.path.isdir(os.path.join(ddir, d))])
    if len(clases) < 2:
        print(f"\u2718 Se necesitan al menos 2 subcarpetas. Encontradas: {clases}")
        return False

    counts = {}
    for c in clases:
        imgs = [f for f in os.listdir(os.path.join(ddir, c))
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        counts[c] = len(imgs)

    total = sum(counts.values())
    _S['dataset_dir'] = ddir
    _S['class_names'] = clases
    _S['img_size'] = img_size

    print(f"\u2714 Dataset: {ddir}")
    print(f"  Clases: {clases}")
    print(f"  Resoluci\u00f3n: {img_size}\u00d7{img_size}")
    print(f"  Total: {total} im\u00e1genes\n")

    max_count = max(counts.values()) if counts else 1
    for c, n in counts.items():
        bar = '\u2588' * max(1, n * 30 // max_count)
        print(f"  {c:>15s} {bar} {n}")

    # Balance check
    if counts:
        vals = list(counts.values())
        if max(vals) > min(vals) * 2:
            print(f"\n  \u26a0 Dataset desbalanceado. Considere capturar m\u00e1s im\u00e1genes de las clases peque\u00f1as.")

    # Preview
    n_cols = min(4, len(clases))
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    if n_cols == 1: axes = [axes]
    for ax, c in zip(axes, clases[:n_cols]):
        cdir = os.path.join(ddir, c)
        imgs = [f for f in os.listdir(cdir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if imgs:
            ax.imshow(load_img(os.path.join(cdir, random.choice(imgs)),
                               target_size=(img_size, img_size)))
        ax.set_title(f'{c} ({counts[c]})', fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.suptitle('Vista previa', fontsize=14)
    plt.tight_layout(); plt.show()
    return True

def _cargar_carpeta(_):
    with w_out_data:
        clear_output(wait=True)
        ddir = w_data_path.value.strip()
        if not os.path.isdir(ddir):
            print(f"\u2718 No se encontr\u00f3: {ddir}")
            return
        _show_dataset_info(ddir, w_imgsize.value)

def _abrir_camara(_):
    with w_out_data:
        clear_output(wait=True)
        raw = w_cam_classes.value.strip()
        if not raw:
            print("\u2718 Escriba los nombres de las clases separados por coma.")
            return
        clases = [c.strip() for c in raw.split(',') if c.strip()]
        if len(clases) < 2:
            print("\u2718 Necesita al menos 2 clases.")
            return

        output_dir = w_cam_output.value.strip()
        tiempo = w_cam_time.value
        cam_idx = w_cam_index.value

        # Ejecutar script de captura
        script = os.path.join('..', 'scripts', 'capturar_clases.py')
        if not os.path.exists(script):
            print(f"\u2718 No se encontr\u00f3 {script}")
            return

        cmd = [sys.executable, script] + clases + [
            '--tiempo', str(tiempo),
            '--salida', output_dir,
            '--camara', str(cam_idx)
        ]
        print(f"Abriendo c\u00e1mara para capturar {len(clases)} clases...")
        print(f"Tiempo por clase: {tiempo}s")
        print(f"\nSe abrir\u00e1 una ventana de OpenCV.")
        print(f"ESPACIO = iniciar captura | q = salir\n")

        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)

        # Verificar resultado
        if os.path.isdir(output_dir):
            _show_dataset_info(output_dir, w_imgsize2.value)

w_btn_folder.on_click(_cargar_carpeta)
w_btn_camera.on_click(_abrir_camara)

tab_folder = widgets.VBox([
    w_data_path, w_imgsize, w_btn_folder
])
tab_camera = widgets.VBox([
    widgets.HTML('<p>Ingrese las clases. Se abrir\u00e1 la c\u00e1mara para capturar cada una con tiempo fijo (dataset balanceado).</p>'),
    w_cam_classes, w_cam_time, w_cam_output, w_cam_index, w_imgsize2,
    w_btn_camera
])

tabs = widgets.Tab(children=[tab_folder, tab_camera])
tabs.set_title(0, 'Carpeta de im\u00e1genes')
tabs.set_title(1, 'Capturar con c\u00e1mara')

display(widgets.VBox([tabs, w_out_data]))

---
## 2. Entrenar

Entrene un modelo nuevo o cargue uno existente (.h5).

Los parámetros avanzados (augmentation, épocas, learning rate) tienen valores por defecto razonables.

In [ ]:
# === Controles ===
w_epochs = widgets.IntSlider(value=30, min=5, max=100, step=5,
                             description='\u00c9pocas m\u00e1x:')
w_lr = widgets.FloatLogSlider(value=0.001, base=10, min=-4, max=-2, step=0.5,
                              description='Learning rate:')
w_rotation = widgets.IntSlider(value=20, min=0, max=90, step=5,
                               description='Rotaci\u00f3n:')
w_shift = widgets.FloatSlider(value=0.1, min=0, max=0.3, step=0.05,
                              description='Desplazamiento:')
w_bright_lo = widgets.FloatSlider(value=0.8, min=0.5, max=1.0, step=0.05,
                                  description='Brillo m\u00edn:')
w_bright_hi = widgets.FloatSlider(value=1.2, min=1.0, max=1.5, step=0.05,
                                  description='Brillo m\u00e1x:')
w_zoom = widgets.FloatSlider(value=0.1, min=0, max=0.3, step=0.05,
                             description='Zoom:')

w_acc_aug = widgets.Accordion(children=[widgets.VBox([
    w_rotation, w_shift, w_bright_lo, w_bright_hi, w_zoom
])])
w_acc_aug.set_title(0, 'Data Augmentation')
w_acc_aug.selected_index = None

w_acc_hp = widgets.Accordion(children=[widgets.VBox([w_epochs, w_lr])])
w_acc_hp.set_title(0, 'Hiperpar\u00e1metros')
w_acc_hp.selected_index = None

w_btn_train = widgets.Button(
    description=' Entrenar', button_style='success', icon='play',
    layout=widgets.Layout(width='200px', height='40px')
)
w_progress = widgets.IntProgress(value=0, min=0, max=100,
                                  description='Progreso:',
                                  layout=widgets.Layout(width='500px',
                                                        visibility='hidden'))

# === Cargar existente ===
w_h5_upload = widgets.FileUpload(accept='.h5,.keras', multiple=False,
                                 description='Subir .h5')
w_h5_path = widgets.Text(value='', placeholder='Ruta al .h5',
                         description='Ruta:',
                         layout=widgets.Layout(width='500px'))
w_btn_load = widgets.Button(
    description=' Cargar .h5', button_style='info', icon='upload',
    layout=widgets.Layout(width='200px', height='40px')
)

w_out_train = widgets.Output()

# --- Callback de progreso ---
class _Progress(tf.keras.callbacks.Callback):
    def __init__(self, bar, out, total):
        self.bar, self.out, self.total = bar, out, total
    def on_epoch_end(self, epoch, logs=None):
        self.bar.value = int((epoch + 1) / self.total * 100)
        with self.out:
            a = logs.get('accuracy', 0)
            va = logs.get('val_accuracy', 0)
            print(f"  \u00c9poca {epoch+1}: train={a:.1%} val={va:.1%}", flush=True)

def _split(ddir, split_dir, names):
    if os.path.exists(os.path.join(split_dir, 'train')): return
    random.seed(42)
    for c in names:
        imgs = sorted([f for f in os.listdir(os.path.join(ddir, c))
                       if f.lower().endswith(('.jpg','.jpeg','.png'))])
        random.shuffle(imgs)
        n = int(len(imgs) * 0.8)
        for sub, lst in [('train', imgs[:n]), ('validation', imgs[n:])]:
            d = os.path.join(split_dir, sub, c)
            os.makedirs(d, exist_ok=True)
            for f in lst:
                dst = os.path.join(d, f)
                if not os.path.exists(dst):
                    shutil.copy(os.path.join(ddir, c, f), dst)

def _build(img_size, n_classes):
    inp = layers.Input(shape=(img_size, img_size, 3))
    x = inp
    for f in [32, 64, 128]:
        x = layers.Conv2D(f, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return models.Model(inputs=inp, outputs=out)

def _entrenar(_):
    with w_out_train:
        clear_output(wait=True)
        if not _S['class_names']:
            print("\u2718 Primero cargue datos (Paso 1)."); return

        ddir = _S['dataset_dir']
        names = _S['class_names']
        sz = _S['img_size']
        nc = len(names)
        ep = w_epochs.value
        lr = w_lr.value

        split_dir = ddir.rstrip('/') + '_split'
        _S['split_dir'] = split_dir
        print("Preparando datos...")
        _split(ddir, split_dir, names)

        tgen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=w_rotation.value,
            width_shift_range=w_shift.value,
            height_shift_range=w_shift.value,
            brightness_range=[w_bright_lo.value, w_bright_hi.value],
            zoom_range=w_zoom.value,
            fill_mode='nearest'
        ).flow_from_directory(
            os.path.join(split_dir, 'train'),
            target_size=(sz, sz), batch_size=32,
            class_mode='categorical', shuffle=True
        )
        vgen = ImageDataGenerator(rescale=1./255).flow_from_directory(
            os.path.join(split_dir, 'validation'),
            target_size=(sz, sz), batch_size=32,
            class_mode='categorical', shuffle=False
        )

        print(f"CNN: {sz}px, {nc} clases, {tgen.samples} train, {vgen.samples} val")
        model = _build(sz, nc)
        model.compile(optimizer=Adam(learning_rate=lr),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        w_progress.value = 0
        w_progress.layout.visibility = 'visible'
        print(f"\nEntrenando (m\u00e1x {ep} \u00e9pocas, LR={lr})...\n")

        h = model.fit(tgen, epochs=ep, validation_data=vgen, verbose=0,
                      callbacks=[
                          EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
                          ReduceLROnPlateau(factor=0.5, patience=5, verbose=1),
                          _Progress(w_progress, w_out_train, ep)
                      ])
        _S['model'] = model
        _S['history'] = h
        w_progress.value = 100

        ne = len(h.history['accuracy'])
        ta = h.history['accuracy'][-1]
        va = h.history['val_accuracy'][-1]
        print(f"\n\u2714 Listo ({ne} \u00e9pocas) — Train: {ta:.1%} | Val: {va:.1%}")
        if ta - va > 0.15:
            print(f"  \u26a0 Posible overfitting (gap {ta-va:.1%})")

        fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
        r = range(1, ne + 1)
        a1.plot(r, h.history['accuracy'], 'b-', label='Train')
        a1.plot(r, h.history['val_accuracy'], 'r-', label='Val')
        a1.set_title('Accuracy'); a1.legend(); a1.grid(alpha=0.3)
        a2.plot(r, h.history['loss'], 'b-', label='Train')
        a2.plot(r, h.history['val_loss'], 'r-', label='Val')
        a2.set_title('Loss'); a2.legend(); a2.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

def _cargar_h5(_):
    with w_out_train:
        clear_output(wait=True)
        path = None
        if w_h5_upload.value:
            info = w_h5_upload.value
            if isinstance(info, dict):
                fname = list(info.keys())[0]
                content = info[fname]['content']
            else:
                fname = info[0].name
                content = info[0].content
            path = f'/tmp/{fname}'
            with open(path, 'wb') as f: f.write(content)
        elif w_h5_path.value.strip():
            path = w_h5_path.value.strip()
        else:
            print("\u2718 Suba o escriba la ruta."); return

        if not os.path.exists(path):
            print(f"\u2718 No existe: {path}"); return

        try:
            model = load_model(path, compile=False)
            model.compile(optimizer='adam', loss='categorical_crossentropy',
                          metrics=['accuracy'])
        except Exception as e:
            print(f"\u2718 Error: {e}"); return

        det_sz = model.input_shape[1]
        if det_sz and det_sz != _S['img_size']:
            _S['img_size'] = det_sz
            print(f"  Resoluci\u00f3n ajustada a {det_sz}")

        n_out = model.output_shape[-1]
        if _S['class_names'] and n_out != len(_S['class_names']):
            print(f"  \u26a0 Modelo: {n_out} salidas, datos: {len(_S['class_names'])} clases")

        _S['model'] = model
        _S['history'] = None
        print(f"\u2714 Modelo cargado: {os.path.basename(path)}")
        print(f"  Input: {model.input_shape} | Salidas: {n_out} | Params: {model.count_params():,}")

w_btn_train.on_click(_entrenar)
w_btn_load.on_click(_cargar_h5)

t_train = widgets.VBox([w_acc_aug, w_acc_hp, w_btn_train, w_progress])
t_load = widgets.VBox([
    widgets.HTML('<p>Cargue un modelo .h5 previamente entrenado:</p>'),
    w_h5_upload, w_h5_path, w_btn_load
])
tabs2 = widgets.Tab(children=[t_train, t_load])
tabs2.set_title(0, 'Entrenar nuevo')
tabs2.set_title(1, 'Cargar .h5 existente')

display(widgets.VBox([tabs2, w_out_train]))

---
## 3. Probar

Evalúe el modelo sobre todo el dataset o pruebe con imágenes individuales.

In [ ]:
def _predict_show(img_display, img_array, titulo=''):
    """Predice y muestra barras de confianza."""
    preds = _S['model'].predict(np.expand_dims(img_array, 0), verbose=0)[0]
    idx = np.argmax(preds)
    conf = preds[idx]
    names = _S['class_names']

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4),
                                  gridspec_kw={'width_ratios': [1, 1.2]})
    a1.imshow(img_display); a1.axis('off')
    c = 'green' if conf > 0.7 else 'orange' if conf > 0.4 else 'red'
    a1.set_title(f'{names[idx]} ({conf:.1%})', fontsize=14,
                 color=c, fontweight='bold')

    colors = ['#2ecc71' if i == idx else '#bdc3c7' for i in range(len(names))]
    bars = a2.barh(names, preds, color=colors)
    a2.set_xlim(0, 1); a2.set_xlabel('Confianza')
    for bar, val in zip(bars, preds):
        a2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f'{val:.1%}', va='center', fontsize=10)
    if titulo: fig.suptitle(titulo, fontsize=11)
    plt.tight_layout(); plt.show()
    return names[idx], conf

In [ ]:
w_btn_eval = widgets.Button(description=' Evaluar dataset',
                            button_style='success', icon='bar-chart')
w_btn_rand = widgets.Button(description=' Imagen aleatoria',
                            button_style='info', icon='random')
w_up_img = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False,
                              description='Subir foto')
w_btn_pred = widgets.Button(description=' Predecir',
                            button_style='warning', icon='eye')
w_out_test = widgets.Output()

def _ready():
    if not _S['model']:
        print("\u2718 Primero entrene o cargue un modelo (Paso 2)."); return False
    if not _S['class_names']:
        print("\u2718 Primero cargue datos (Paso 1)."); return False
    return True

def _eval_dataset(_):
    with w_out_test:
        clear_output(wait=True)
        if not _ready(): return
        ddir = _S['dataset_dir']; names = _S['class_names']; sz = _S['img_size']

        gen = ImageDataGenerator(rescale=1./255).flow_from_directory(
            ddir, target_size=(sz, sz), batch_size=32,
            class_mode='categorical', shuffle=False, classes=names)

        print(f"Evaluando {gen.samples} im\u00e1genes...")
        loss, acc = _S['model'].evaluate(gen, verbose=0)
        gen.reset()
        yp = _S['model'].predict(gen, verbose=0)
        y_pred = np.argmax(yp, axis=1)
        y_true = gen.classes
        labels = list(gen.class_indices.keys())

        print(f"\nAccuracy: {acc:.1%}\n")
        print(classification_report(y_true, y_pred,
                                    target_names=labels, zero_division=0))

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(max(5, len(labels)*1.5), max(4, len(labels)*1.2)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels)
        plt.xlabel('Predicci\u00f3n'); plt.ylabel('Real')
        plt.title(f'Accuracy: {acc:.1%}')
        plt.tight_layout(); plt.show()

        # Mostrar errores
        errs = [(i, gen.filenames[i]) for i in range(len(y_true))
                if y_pred[i] != y_true[i]]
        if errs:
            n = min(8, len(errs))
            print(f"\nErrores: {len(errs)} de {len(y_true)}")
            fig, axes = plt.subplots(2, 4, figsize=(16, 8))
            for ax in axes.ravel(): ax.axis('off')
            for j, ax in enumerate(axes.ravel()):
                if j >= n: break
                idx, fname = errs[j]
                p = os.path.join(ddir, fname)
                if os.path.exists(p):
                    ax.imshow(load_img(p, target_size=(sz, sz)))
                    ax.set_title(f"Real: {labels[y_true[idx]]}\n"
                                 f"Pred: {labels[y_pred[idx]]} ({yp[idx][y_pred[idx]]:.0%})",
                                 fontsize=9, color='red')
            plt.suptitle('Errores', fontsize=14)
            plt.tight_layout(); plt.show()

def _rand_img(_):
    with w_out_test:
        clear_output(wait=True)
        if not _ready(): return
        imgs = []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            imgs += glob.glob(os.path.join(_S['dataset_dir'], '**', ext), recursive=True)
        if not imgs: print("No hay im\u00e1genes."); return
        p = random.choice(imgs)
        real = os.path.basename(os.path.dirname(p))
        img = load_img(p, target_size=(_S['img_size'], _S['img_size']))
        pred, conf = _predict_show(img, img_to_array(img)/255.0,
                                    titulo=f'Real: {real}')
        ok = '\u2714' if pred == real else '\u2718'
        print(f"{ok} Real: {real} | Pred: {pred} ({conf:.1%})")

def _pred_upload(_):
    with w_out_test:
        clear_output(wait=True)
        if not _ready(): return
        if not w_up_img.value: print("\u2718 Suba una imagen."); return
        info = w_up_img.value
        content = (list(info.values())[0]['content']
                   if isinstance(info, dict) else info[0].content)
        from PIL import Image
        sz = _S['img_size']
        pil = Image.open(io.BytesIO(content)).convert('RGB').resize((sz, sz))
        arr = np.array(pil).astype('float32') / 255.0
        pred, conf = _predict_show(pil, arr, titulo='Imagen subida')
        print(f"Predicci\u00f3n: {pred} ({conf:.1%})")

w_btn_eval.on_click(_eval_dataset)
w_btn_rand.on_click(_rand_img)
w_btn_pred.on_click(_pred_upload)

display(widgets.VBox([
    w_btn_eval,
    widgets.HTML('<hr>'),
    w_btn_rand,
    widgets.HTML('<br>'),
    widgets.HBox([w_up_img, w_btn_pred]),
    w_out_test
]))

---
## 4. Exportar

Guarde el modelo y obtenga la configuración para `scripts/prueba_video.py` (cámara en vivo).

In [ ]:
w_fname = widgets.Text(value='mi_modelo.h5', description='Nombre:',
                       layout=widgets.Layout(width='400px'))
w_btn_save = widgets.Button(description=' Guardar', button_style='success',
                            icon='download',
                            layout=widgets.Layout(width='200px', height='40px'))
w_out_export = widgets.Output()

def _exportar(_):
    with w_out_export:
        clear_output(wait=True)
        if not _S['model']:
            print("\u2718 No hay modelo."); return
        f = w_fname.value.strip()
        if not f.endswith(('.h5','.keras')): f += '.h5'
        _S['model'].save(f)
        mb = os.path.getsize(f) / 1024 / 1024
        print(f"\u2714 Guardado: {f} ({mb:.1f} MB)")

        names = _S['class_names']
        sz = _S['img_size']
        print(f"\n{'='*50}")
        print("CONFIG PARA scripts/prueba_video.py:")
        print(f"{'='*50}")
        print(f'MODEL_PATH = \"labs/{f}\"')
        print(f'IMG_SIZE = {sz}')
        print(f'CLASS_NAMES = {names}')
        print(f'CONFIDENCE_THRESHOLD = 0.6')
        print(f"{'='*50}")

w_btn_save.on_click(_exportar)
display(widgets.VBox([w_fname, w_btn_save, w_out_export]))